# U4-L2：走近来宾（距离估计与核对）

**课程目标**

1. 能区分里程计估计的行进量与机器人到来宾的间距，用卷尺记录实际行进量和最终间距。
2. 能在 `walk` 模式下清零里程计、按计划行进量停止，并在相同测量条件下比较不同前进速度的停靠结果。
3. 能用框高占比进行距离标定与独立核对，根据实测记录选择停止阈值，并说明阈值适用的目标、拍摄条件和距离范围。

## 一、本课要解决的问题

上节课结束时，我们具备了调整机器人，使得机器人面向目标物方位（目标物在身体正前方、且处于摄像头画面中心）的能力。将检测目标的选择由“瓶子”换为“人”就实现了“转向来宾”的任务目标。此时机器人不但已经“正面”来宾，也知道自己转到了哪个角度。但是它还站在原地，和来宾之间的距离几乎没有改变。

本课的核心任务是让机器人走近来宾，机器人需要决定向前走多远。本课中来宾保持站定，先用卷尺测机器人与目标的间距，再计算计划行进量。

起步间距确定后，还要核查机器人按里程计读数停下时，实际走了多远、离来宾还有多远。相机中的目标大小可以提供另一条距离线索。

在展厅等场景下机器人进行迎宾，需要能够稳定走向、并停在来宾面前。根据K1机器人的身高，最终间距 1.2 米左右比较合适——太远显得冷淡，太近不好实现“有目光接触的交流”。

对这个项目目标，我们进行一下简化：目标停靠间距设为 1.2 米，允许误差为 0.2 米。对于这个目标，本课的核心问题：

> **机器人靠什么知道自己走了多远（需要判断自己实际走过的距离），又怎么判断已经走到位了（还有没有其他可以用于判定距离的感知数据）？**

## 二、里程计里的距离

要控制行进距离，需要能获取机器人在不同时刻的位置（要在坐标空间中对机器人的位置进行估计）。里程计就是为了这个任务而设计的，BoosterOS SDK 通过里程计接口返回位置和朝向；这些值是系统估算结果，不是卷尺测量值。


### 2.1 里程计中的位置

BoosterOS SDK 中 `get_odom().pose_2d` 返回里程计参考坐标中的 横坐标x、纵坐标y 和朝向角，单位分别是米、米和弧度。x、y 表示估计位置，朝向角表示机器人面向哪个方向。

可以把里程计估计类比为“数步子”：根据机器人运动的信息推算位置变化。实际系统还要处理身体姿态等因素，不能把它简单等同于步数乘以固定步长。

思考一下：只有 x、y，没有朝向角，能不能准确表达机器人在空间中的位置？

**STEP 01：** 连接机器人，读一次里程计

In [ ]:
from boosteros.robots.booster import BoosterRobot

robot = BoosterRobot()

print("已连接：", robot.robot_info.model, robot.robot_info.serial_number)
print("当前模式：", robot.get_mode())

odom = robot.get_odom()
print("里程计返回值：", odom)

x, y, yaw = odom.pose_2d
print(f"位置：x = {x:.3f} m，y = {y:.3f} m")
print(f"方向：{yaw:.3f} rad")

### 2.2 清零与累计

刚才读到的 x，是机器人相对当前里程计参考原点的 x 方向位置，不是开机后累计走过的路程。要方便地记录“这一次向前移动了多少”，可先清零。清零不改变机器人当前站在哪里，只是为了清完之后x、y读数从零起步。里程计清零后，机器人基本沿参考坐标的 x 轴直行时，x 的变化可近似作为前向位移。转弯或明显偏航时，x 只是这个方向上的位置变化，不能代表整段路程。

`reset_odom()` 把当前位置和朝向设为里程计的参考零位，机器人本身不会移动。清零以后的位置读数，都相对于这个参考位姿表示。

**STEP 02：** 清零，再读一次

<font color="red">提示：</font>下一格将执行 `robot.reset_odom()`，该清零接口要求机器人处于 `walk` 行走模式。因此运行下一格程序前，需要将机器人切换到 `walk` 行走模式。通过按键切换即可。

In [ ]:
robot.reset_odom()

x, y, yaw = robot.get_odom().pose_2d
print(f"清零之后：x = {x:.3f} m，y = {y:.3f} m，方向 = {yaw:.3f} rad")

<font color="red">注意：</font>实际上，刚刚这格程序每次我们运行后的输出都会略有不同，如下展示三次输出结果：
- 结果1：`清零之后：x = -0.003 m，y = 0.002 m，方向 = 0.058 rad`
- 结果2：`清零之后：x = 0.000 m，y = 0.000 m，方向 = 0.002 rad`
- 结果3：`清零之后：x = -0.000 m，y = 0.000 m，方向 = -0.000 rad`

清零后到读取之间，估计值仍可能更新，因此输出不一定恰好为零。记录本次 x、y 和朝向；偏差是否可接受，要结合计划行进量和朝向要求判断。反复清零直到显示为零，不能证明里程计已经准确。

### 2.3 还差多少

里程计清零后，判断「走到了没有」就是一个减法：目标距离减去当前读数。结果还是正数，说明没到；接近 0，说明到了。

```
还差多少 = 目标距离 − 当前走过的距离
```

里程计行进量和来宾间距是两个不同的量。假设机器人起步时离来宾 2.0 米，目标停在 1.2 米处，目标距离为 `2.0 - 1.2 = 0.8` 米。

如果目标距离是 0.8 米、当前里程计估计走了 0.35 米，那么还差 `0.8 - 0.35 = 0.45` 米。这个计算用于控制行进量。


**STEP 03：** 先只算一次「还差多少」，不发指令

In [ ]:
start_gap = 2.0                 # 示例输入：假设卷尺测得起步间距为 2.0 m
target_gap = 1.2                 # 目标停靠间距（米）
travel_target = start_gap - target_gap
x, y, yaw = robot.get_odom().pose_2d            # 假设示例：里程计估计的行进量（米）
remain = travel_target - x
print(f"目标 {travel_target:.2f} m，已估计行进 {x:.2f} m，还差 {remain:.2f} m")

## 三、里程计读数偏差与漂移

如果里程计的读数一直十分精准，这一课到这儿就结束了。问题是它不准。


### 3.1 读数偏差与漂移

里程计提供的是估计值。它和实际位移之间可能有偏差，需要用卷尺测量结果核对。

这和上一课计算偏差原理相同：每一轮读一次当前值，算一次还差多少，再决定这一轮发什么指令。差别是上一课读的是画面里的框，这一课读的是里程计。

要测量偏差，需要在地面标记起点和终点，并用卷尺测量。两次测量应选择机身同一个参考点（比如脚尖），避免把起止点选择的差异算成里程计误差。地面、运动速度、行进方向及测量参考点都会影响比较结果。记录时保留这些条件，不能只凭一次偏差给所有行走任务加上同一个修正值。

如果位置估计的误差在运动中不断累积，估计位置会逐渐偏离实际位置，这种现象通常称为漂移。一次停靠的差值只能描述这次结果；要观察误差是否随行进距离增加，还需在不同距离下重复测量。

此外，需要注意的是目前我们在假定机器人面向目标行进，也就是前进（只需要评估x），而实际上机器人行进时也可能产生左右偏差。

### 3.2 按里程计估计行进量

确保机器人已经进入 `walk` 模式，然后清零里程计。卷尺测得起步间距为 `start_gap`，目标间距为 1.2 米，两者相减得到计划行进量 `target_travel`。

程序每轮读取里程计 x 分量，计算还需行进多少，再发送前进速度；读数接近计划值时发送零速度指令。这是按里程计估计值停止，实际位移和最终间距仍需测量。

**STEP 04：** 实际行走，并记录停止位置

<font color="red">提示：</font>
1. 本格会让机器人前进。先确认机器人与目标之间无人经过，机器人运行时在侧方看护，遇到问题及时将机器人切换到 `prepare` 模式。
2. 停下后测量起止标记之间的实际前向位移，以及机器人与来宾的最终间距。

In [ ]:
import time

# start_gap = float(input("输入卷尺测得的起步间距（米）："))
start_gap = 2.0                  # 采用固定值，如果想自行输入的话，取消上一行注释，并把本行注释掉。
target_gap = 1.2                 # 目标停靠间距（米）
target_travel = start_gap - target_gap

robot.reset_odom()               # SDK 要求在 walk 模式调用
x = 0.0

for i in range(200):
    x, y, yaw = robot.get_odom().pose_2d
    remain = target_travel - x
    if remain <= 0.02:
        print(f"第 {i + 1} 轮接近计划行进量：里程计估计 {x:.3f} m")
        break
    robot.set_velocity(0.3, 0, 0)
    print(f"第 {i + 1} 轮：已走 {x:.3f} m，还差 {remain:.3f} m")
    time.sleep(0.2)

robot.set_velocity(0, 0, 0)

print(f"已停止。里程计估计行进量 {x:.3f} m；最终间距请用卷尺测量")


### 3.3 读数偏差怎么记录

比较时，用里程计估计的行进量减去卷尺测得的实际行进量，记录差值及正负。

单次差值描述这次的偏差；多次实验可以比较偏差的大小和方向。

要判断漂移是否随行进距离增加，还需在不同距离下重复测量。


### 3.4 重复行进测定合理的行进速度参数

实验目标：保持起点、目标站位、行进方向和地面条件相近，每次只调整前进速度 `vx`，比较里程计读数、实际行进量和最终间距，选择在本次条件下更合适的速度。一个速度在本次实验中偏差较小，不代表能消除里程计漂移。

注意：`reset_odom()` 只清零读数，不会让机器人回到起点，注意复位机器人位置时候的操作安全，`walk` 模式下搬动机器人比较危险！

每次记录起步间距、计划行进量、设定速度、程序输出的末次 x、卷尺实际前向位移和最终间距。程序中的 x 来自循环末次读取，可能早于停稳，记录时须保留这个区别。先由教师确认速度范围，每次复位后重新测量起步间距。

**STEP 05：** 按计划行进量停止，记录数据

In [ ]:
start_gap_compare = 2.0    # 实验卷尺测得的起步间距（米）。
target_gap = 1.2           # 计划机器人停止时距离目标物的距离（米）。
target_travel = start_gap_compare - target_gap

robot.reset_odom()
x = 0.0

for i in range(200):
    x, y, yaw = robot.get_odom().pose_2d
    remain = target_travel - x
    if remain <= 0.02:
        print(f"第 {i + 1} 轮：里程计估计行进量 {x:.3f} m")
        break
    robot.set_velocity(0.3, 0, 0)   # 本实验的目标时多次测量，确定set_velocity(vx, vy, vyaw)中合理的vx
    time.sleep(0.2)

robot.set_velocity(0, 0, 0)

print(f"已停下。里程计估计行进量 {x:.3f} m；最终间距请用卷尺测量")

## 四、框大小的距离标定

除了里程计估计（数步子），机器人相机视角下图像中目标的大小也能提供一条距离线索。


### 4.1 检测框大小与距离

同一个目标，目标与机器人朝向不变时（头部没有俯仰角和偏航角），距离变近，完整目标在画面中的面积通常增大。本课用“检测框高度 ÷ 图像高度”，采用高度占比表示检测框大小。

U3-L4 用框大小区分近、中、远。本课加入卷尺测得的距离，尝试找到框高占比换算为米的公式。

这一换算需要检测框始终对应相同的身体范围。靠近后若头部或脚部离开画面，检测框不再表示相同范围，框高可能接近画面高度却不再随距离正常变化。此时应记录“目标不完整，估距条件不成立”，停止使用该次框高估距。


### 4.2 已知距离上的标定

标定是在已知条件下建立读数与实际量之间的对应。本实验先用卷尺量出 1.2 米间距，记录这时的框高占比；随后在 1.5 米处检查换算结果。

> 提示：为了便于开展实验，可以灵活根据场地大小选择不同的目标物与距离进行标定测量。掌握原理更重要。

这两个位置都必须能看到同一位来宾的完整身体。如果 1.2 米处无法满足这一条件，先记录画面情况并调整实验距离，不继续套用下面固定为 1.2 米的换算代码。

取图时保持机器人处于站姿，记录头部朝向，后续行走实验要保持头部角度，如果头部角度改变，需要在新的拍摄姿态下重新标定。

**STEP 06：** 在已知距离上记录框高占比

<font color="red">提示：</font>
1. 本格只取图，不发送运动指令，但是为了维持前后测量姿态一致，且保持安全，可以在 `prepare` 模式下进行；也可以选择 `walk` 模式，但是此时一定要严格注意移动机器人位置时候的安全问题。
2. 画面中只安排同一位站定的来宾（也可以置换为其他目标物体），用卷尺测量至 1.2 米处。
  - 先检查图像能否容纳来宾完整身体；条件不满足时记录情况，并调整距离。
  - 后面的 1.5 米检查保持机器人站姿、遮挡和拍摄姿态一致。


STEP 06-1：先读图（为了确认目标在图中）

In [ ]:
from boosteros.brain import Detection
import matplotlib.pyplot as plt

detector = Detection(model="person")

img = robot.get_image(img_type="rgb")
img_array = img.to_numpy()
print("画面尺寸：", img.width, "×", img.height)

plt.imshow(img_array[:, :, ::-1])      # [:, :, ::-1] 把采集到的 bgr 通道转为 rgb 排列
plt.show()

STEP 06-2：检测一下

In [ ]:
results = detector.detect(img_array)
print("检测到", len(results), "个目标")
h_ratio_1m2 = None

if results:
    r = results[0]  # 本实验只安排一位来宾；代码取第一个检测框，不识别来宾身份
    h_ratio = r.bbox.height / img.height
    h_ratio_1m2 = h_ratio
    print(f"{r.class_name}  置信度 {r.confidence:.2f}")
    print(f"框高 {r.bbox.height} 像素，占画面高度 {h_ratio:.3f}")
else:
    print("没有检测到目标，请调整位置后重新取图")

记录一次真实的测试结果：
```
检测到 1 个目标
person  置信度 0.94
框高 382 像素，占画面高度 0.853
```

上述测试结果的测试环境是：人位于机器人正前方1.2米处的测量结果，实际测量结果还会与人的身高等多个因素相关。

STEP 06-3：看一下带检测框的图像

In [ ]:
detector.plot(img_array, results)

### 4.3 从框高占比估算距离

把已知距离记为 `d0`，对应框高占比记为 `r0`，当前框高占比记为 `r`。同一位来宾在完整可见、拍摄条件相近时，可作近似换算：

```
估计距离 = d0 × r0 ÷ r
```

例如，标定距离为 1.2 米，当前框高占比是标定时的一半，则估计距离为 2.4 米。这是用于解释公式的假设数据。当前占比增大时，估计距离会减小，符合近大远小的变化方向。

这里使用高度占比，没有使用检测框面积。身高、站姿、遮挡、透视和检测框抖动都会影响结果，必须与卷尺读数核对。目标不完整时，前面的近似关系不成立。


**STEP 07：** 用第二个已知距离检查估算（只读图，不发指令）

<font color="red">提示：</font>画面中仍只保留同一个目标。目标到机器人的距离调整为 1.5 米，重新用卷尺测量并取图。不要复用 1.2 米处的旧画面；估算值与卷尺实测值一起记录。

In [ ]:
img_check = robot.get_image(img_type="rgb")
results_check = detector.detect(img_check.to_numpy())
if results_check:
    box_height = results_check[0].bbox.height
    h_ratio_now = box_height / img_check.height
    actual_gap = float(input("输入卷尺测得的当前间距（米）："))
    distance_est = 1.2 * (h_ratio_1m2 / h_ratio_now)
    error = distance_est - actual_gap
    print(f"框高估算 {distance_est:.2f} m，卷尺实测 {actual_gap:.2f} m")
    print(f"本次估算误差 {error:+.2f} m")
else:
    print("画面里没有目标，无法估算；请调整位置后重新取图") 

### 4.3.1 估距结果与工程选择

STEP 07 用 §4.3 的公式计算距离，再与卷尺结果核对：

```
估计距离 = d0 × r0 ÷ r
```

记 `k = d0 × r0`，这个式子可以写成 `distance = k ÷ h_ratio`，表达的是反比例近似：框高占比增大，估计距离减小。在拍摄条件相近、同一目标完整可见时，这符合近大远小的变化方向。

如果 1.5 米处的估计有明显偏差，先检查目标是否完整、头部角度和站姿是否变化、检测框是否抖动，以及卷尺参考点是否一致。一次核对出现偏差，不能据此认定两者应改用正比例或某个多项式关系。误差大小以本组实际记录为准。

工程上可按任务需要选择处理方式：

- **需要输出距离数值**：采集多个距离的对应数据，再用未参与标定的数据检查误差。数据拟合或机器学习也是可研究的办法，但仍需要独立测试，不能保证换一种算法就一定更准确。
- **只需判断是否已接近指定位置**：在目标间距附近多次记录框高占比，选择一个停止阈值，再用实际行走和卷尺检查停靠结果。此时程序判断的是框大小是否达到条件，不必每轮换算出米数。

接下来的任务采用第二种方式。阈值仍受目标尺寸、视野和拍摄姿态限制，需要实测核对。

### 4.4 重新标定的条件

本课使用 `Detection(model="person")`，检测对象是人。

为了快速掌握背后的知识、技术原理，你也可以选择一个更小的目标物，比如瓶子，在空间更小的场地下完成实践。

需要注意的是：更换来宾（换不同的人）、明显改变站姿、相机高度或头部朝向后，原来的框高比例可能不再适用，需要重新标定。

更换成其他目标类别时，检测模型和标定数据都要重新检查。人形的标定值不能直接用于瓶子等其他目标。


**STEP 08：自主编写按框高阈值停止的程序。** 选择目标，在目标间距及稍远位置分别多次取图，记录“框高 ÷ 图像高度”，再确定同一尺度下的停止阈值。若仍以迎宾为任务，就用同一位来宾检查 1.2 ± 0.2 米的实际停靠结果；改用瓶子时另定实验间距并重新标定。

先只打印“继续”或“停止”核对分支。目标丢失、身体不完整或数据无效时，应停止本轮，不能按“未达到阈值”继续前进。获得教师对速度、路线和停止条件的确认后，再进行行走测试；程序须有轮数或时间上限，正常退出和中断时均应停止运动。只使用已学的取图、检测、循环和条件判断即可。

逐轮重新取图；框高占比达到或超过阈值就发送零速度。机器人停稳后，用卷尺测最终间距，在 §5.3 记录判断与实际落点。阈值是否合适由停靠记录检验。

In [ ]:
# 根据前面所学，自主设计程序，完成任务



## 五、观察、记录与检查

### 5.1 里程计行进与速度比较

STEP 04 记录一次实际行走，STEP 05 比较不同 `vx`。每次恢复起点后重新量起步间距；测量起止位置都使用机身同一个参考点。距离单位为米，速度单位为米/秒。

| 实验/轮次 | 起步间距 | 计划行进量 | 设定 vx | 循环末次 x（程序输出） | 实际前向位移（卷尺） | 停稳后间距（卷尺） | 退出原因/是否停稳 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| STEP 04 | | | | | | | |
| STEP 05，第1次 | | | | | | | |
| STEP 05，第2次 | | | | | | | |
| STEP 05，第3次 | | | | | | | |

按记录计算：

```
计划行进量 = 起步间距 − 目标间距
本次读数与实测差值 = 循环末次 x − 卷尺实际前向位移
停靠误差 = 停稳后间距 − 目标间距
```

停靠误差为正，表示停得比目标间距更远；为负，表示更近。循环末次 x 可能早于机器人停稳，因此它与卷尺位移的差值还可能包含读数时刻、起停过程和测量方法的影响，不能全部归为里程计算法的误差。

本组在相近条件下选择的 `vx` 是________，依据的轮次和数据是________。若重复结果波动较大，先记录波动和待核条件，不直接宣布某个速度“最准确”。单次结果也不能证明长期漂移已解决。

### 5.2 框高标定与第二距离核对

将 STEP 06 的标定结果和 STEP 07 的新画面结果分别记录。默认标定距离为 1.2 米、核对距离为 1.5 米；若因视野或目标改变了距离，填写实际值，并同步修改公式和代码中的标定距离。

| 用途/步骤 | 实际目标 | 卷尺间距（米） | 头部朝向/站姿 | 目标是否完整 | 框高（像素）/图像高度（像素） | 框高占比 r | 估计间距（米） | 估计值减实测值（米） |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| STEP 06 标定 | | | | | | | 不作独立核对 | 不适用 |
| STEP 07 核对 | | | | | | | | |
| 补测（按需） | | | | | | | | |

标定点用于确定 `d0 × r0`，不能再用这个点本身证明公式准确。核对时重新取图，保持同一目标与拍摄条件；目标不完整或没有有效检测框时，估计间距填“条件不成立”，并写明原因。

本次核对误差是________。下一次需要检查的条件是________。如结果满足本组用途，也按实际结果记录，无需刻意制造较大误差。

### 5.3 停止阈值与自主程序验证

先为 STEP 08 确定条件：目标________；目标间距________米；检测模型/类别________；拍摄姿态________。本课用框高占比作为阈值，不能把像素高度或置信度直接填入占比阈值。

**静态采样：** 在目标间距及稍远处多次取图，观察读数波动。同一距离可增加多行。

| 采样位置 | 卷尺间距（米） | 框高占比 | 完整可见/是否有效 | 准备判为继续还是停止 |
| --- | --- | --- | --- | --- |
| 目标间距，第1次 | | | | |
| 目标间距，第2次 | | | | |
| 稍远位置，第1次 | | | | |
| 稍远位置，第2次 | | | | |

选定阈值________；选择依据________。若两处读数重叠，单一阈值难以稳定区分，应先核查视野、目标姿态和检测框波动，再决定调整或补测。

**程序与行走核对：** 教师确认后再进行运动。先用静态场景检查“未达到阈值、达到阈值、未检出目标”的分支；无法有效检测时应停止，不继续向前。

| 轮次/程序版本 | vx / 阈值 | 起步间距（米） | 程序退出原因 | 最后有效框高占比 | 停稳后间距（米） | 是否符合本次间距要求 | 下一次调整项 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| | | | | | | | |
| | | | | | | | |

因轮数或时间上限、目标丢失而退出时，如实填写，不能记为“达到距离”。阈值来自测量，但实际落点还受取图、检测、发指令和停稳过程影响，最终间距仍由卷尺核对。用瓶子完成的结果，不直接作为来宾停靠任务已完成的依据。

实验结束后先按按钮流程切回准备模式，等机器人站稳，再按教师安排人工支撑、切阻尼并扶坐。恢复起点前检查机器人状态，不在行走模式下直接搬动机器人。

### 5.4 检查点

- 能区分计划行进量、里程计 x、卷尺前向位移和最终间距，并指出各自数据来源。
- 能用相近条件下的重复记录解释速度选择，不把一次偏差推广为长期规律。
- 能说明 `d0 × r0 ÷ r` 的变化方向，以及标定点和独立核对点的不同用途。
- 能根据采样选择框高占比阈值，指出程序中的继续、停止及无效数据处理条件。
- 能用实际间距检验阈值程序，并说明换目标、改变拍摄姿态或目标不完整时为何需要重新检查。

### 5.5 思考题

1. 起步间距为 2.0 米、目标间距为 1.2 米。循环末次 x 为 0.80 米，停稳后卷尺间距为 1.30 米。计划行进量和停靠误差分别是多少？这些数值能否证明机器人实际走了 0.80 米？
2. 同一目标的框高占比从 0.40 增大到 0.50，按本课估距公式，估计距离如何变化？若读数变化来自脚部被画面裁切，还能沿用这个判断吗？
3. 换成更高的来宾后，原阈值程序仍能正常退出，是否就说明停靠间距符合要求？应补充什么测量？
4. 比较 STEP 04 与自己编写的 STEP 08：各自用什么读数决定停止？起步间距未知时，哪些判断仍可使用，又分别依赖哪些条件？